In [1]:
import tensorflow as tf
import numpy as np

from keras.src.metrics.accuracy_metrics import accuracy

from src.model_loader import ModelLoader
from src.data_loader import DataLoader

print(tf.__version__)

2.16.2


# Architecture du modèle

In [2]:
data_loader = DataLoader()
train_full, val_full, test_full = data_loader.load_binary_dataset(positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"])

2025-04-08 15:23:34.805581: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-04-08 15:23:34.805612: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-04-08 15:23:34.805618: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-04-08 15:23:34.805635: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-08 15:23:34.805645: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
cnn_hard_loader = ModelLoader(model_name="cnn_hard_model")
cnn_hard_model = cnn_hard_loader.create_model_CNN_hard()
with tf.device("/gpu:0"):
    cnn_hard_history = cnn_hard_model.fit(train_full, epochs=5, verbose=1, validation_data=val_full, callbacks=[cnn_hard_loader.get_tensorboard_callback()])

/Users/tanguydumontier/PycharmProjects/CESI_DS/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
res_net_loader = ModelLoader(model_name="res_net_model")
res_net_model = res_net_loader.create_model_resnet50()
with tf.device("/gpu:0"):
    res_net_history = res_net_model.fit(train_full, epochs=5, verbose=1, validation_data=val_full, callbacks=[res_net_loader.get_tensorboard_callback()])

In [7]:
inception_loader = ModelLoader(model_name="inception_model")
inception_model = inception_loader.create_model_with_inception(show_summary=False)
with tf.device("/gpu:0"):
    inception_history = inception_model.fit(train_full, epochs=5, verbose=2, validation_data=val_full, callbacks=[inception_loader.get_tensorboard_callback()])

Epoch 1/5


2025-04-08 15:41:37.811329: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


400/400 - 95s - 239ms/step - accuracy: 0.9158 - loss: 0.3194 - val_accuracy: 0.9678 - val_loss: 0.1151
Epoch 2/5


2025-04-08 15:43:12.104196: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


400/400 - 94s - 235ms/step - accuracy: 0.9528 - loss: 0.1815 - val_accuracy: 0.9690 - val_loss: 0.1020
Epoch 3/5


...

400/400 - 90s - 225ms/step - accuracy: 0.9547 - loss: 0.1517 - val_accuracy: 0.9715 - val_loss: 0.0925
Epoch 4/5


2025-04-08 15:46:14.413668: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


400/400 - 89s - 223ms/step - accuracy: 0.9620 - loss: 0.1178 - val_accuracy: 0.9700 - val_loss: 0.0903
Epoch 5/5


2025-04-08 15:47:43.432457: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


400/400 - 88s - 221ms/step - accuracy: 0.9653 - loss: 0.1042 - val_accuracy: 0.9694 - val_loss: 0.0921


In [8]:
inception_model.evaluate(test_full, verbose=2)

125/125 - 20s - 158ms/step - accuracy: 0.9622 - loss: 0.1178


In [6]:
from matplotlib import pyplot as plt

def showTrainingHistory(history: tf.keras.callbacks.History):
    plt.figure()
    plt.plot(history.history['accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper right')
    plt.show()

    plt.figure()
    plt.plot(history.history['loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper right')
    plt.show()

showTrainingHistory(history_res)

NameError: name 'history_res' is not defined

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix(model, x_test, y_test, class_names=None, normalize=False):
    # Prédictions : on obtient les classes prédites (argmax si probabilités/logits)
    y_pred = model.predict(x_test)
    if y_pred.shape[-1] > 1:
        y_pred = np.argmax(y_pred, axis=1)
    else:
        y_pred = (y_pred > 0.5).astype("int32").flatten()  # Cas binaire

    # Si y_test a la forme (N, 1), on le "flatten"
    y_true = y_test.flatten() if len(y_test.shape) > 1 else y_test

    # Génération de la matrice
    cm = confusion_matrix(y_true, y_pred, normalize='true' if normalize else None)

    # Affichage
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Matrice de confusion" + (" normalisée" if normalize else ""))
    plt.show()

125/125 - 4s - 30ms/step - accuracy: 0.9947 - loss: 0.0542
0.054213497787714005 0.9947474002838135


In [1]:
from src.data_loader import DataLoader

data_loader = DataLoader()
train, val, test = data_loader.load_binary_dataset(positive_class="Photo", negative_classes=["Sketch", "Painting"])

2025-04-08 15:13:54.924017: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-04-08 15:13:54.924042: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-04-08 15:13:54.924046: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-04-08 15:13:54.924060: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-08 15:13:54.924068: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>